# 02. Diagnóstico de Qualidade (Bronze)
Verificação dos critérios de governança: **completude, unicidade, consistência, acurácia, integridade referencial e outliers**, nas tabelas usadas pelo objetivo. Os problemas encontrados aqui orientam as transformações que ocorrerão na etapa Silver.

In [0]:
from pyspark.sql import functions as F

tabelas = ["orders", "order_items", "products", "category_translation"]
linhas = []
for t in tabelas:
    df = spark.table(f"mvp_olist.bronze.{t}").drop("_ingestao_timestamp", "_arquivo_origem")
    total = df.count()
    nulos = df.agg(*[F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
                     for c in df.columns]).collect()[0].asDict()
    linhas += [(t, c, total, n, round(100 * n / total, 2)) for c, n in nulos.items()]

df_nulos = spark.createDataFrame(linhas, "tabela string, coluna string, total_linhas long, qtd_nulos long, pct_nulos double")
display(df_nulos.filter("qtd_nulos > 0").orderBy(F.desc("pct_nulos")))

tabela,coluna,total_linhas,qtd_nulos,pct_nulos
orders,order_delivered_customer_date,99441,2965,2.98
products,product_category_name,32951,610,1.85
products,product_name_lenght,32951,610,1.85
products,product_description_lenght,32951,610,1.85
products,product_photos_qty,32951,610,1.85
orders,order_delivered_carrier_date,99441,1783,1.79
orders,order_approved_at,99441,160,0.16
products,product_weight_g,32951,2,0.01
products,product_length_cm,32951,2,0.01
products,product_height_cm,32951,2,0.01


In [0]:
%sql
SELECT 'orders: pedidos duplicados' AS regra, COUNT(*) - COUNT(DISTINCT order_id) AS qtd FROM mvp_olist.bronze.orders
UNION ALL
SELECT 'order_items: itens duplicados', COUNT(*) - COUNT(DISTINCT order_id, order_item_id) FROM mvp_olist.bronze.order_items
UNION ALL
SELECT 'products: produtos duplicados', COUNT(*) - COUNT(DISTINCT product_id) FROM mvp_olist.bronze.products
UNION ALL
SELECT 'itens sem pedido correspondente', COUNT(*) FROM mvp_olist.bronze.order_items i
  LEFT ANTI JOIN mvp_olist.bronze.orders o ON i.order_id = o.order_id
UNION ALL
SELECT 'itens sem produto correspondente', COUNT(*) FROM mvp_olist.bronze.order_items i
  LEFT ANTI JOIN mvp_olist.bronze.products p ON i.product_id = p.product_id
UNION ALL
SELECT 'categorias sem tradução', COUNT(DISTINCT p.product_category_name) FROM mvp_olist.bronze.products p
  LEFT ANTI JOIN mvp_olist.bronze.category_translation t ON p.product_category_name = t.product_category_name
  WHERE p.product_category_name IS NOT NULL

regra,qtd
orders: pedidos duplicados,0
order_items: itens duplicados,0
products: produtos duplicados,0
itens sem pedido correspondente,0
itens sem produto correspondente,0
categorias sem tradução,2


In [0]:
%sql
SELECT
  COUNT(*) AS total_pedidos,
  SUM(CASE WHEN try_to_timestamp(order_purchase_timestamp) IS NULL THEN 1 ELSE 0 END) AS data_compra_invalida,
  MIN(try_to_timestamp(order_purchase_timestamp)) AS primeira_compra,
  MAX(try_to_timestamp(order_purchase_timestamp)) AS ultima_compra,
  SUM(CASE WHEN try_to_timestamp(order_delivered_customer_date) < try_to_timestamp(order_purchase_timestamp) THEN 1 ELSE 0 END) AS entrega_antes_da_compra
FROM mvp_olist.bronze.orders;

total_pedidos,data_compra_invalida,primeira_compra,ultima_compra,entrega_antes_da_compra
99441,0,2016-09-04T21:15:19.000Z,2018-10-17T17:30:18.000Z,0


In [0]:
%sql
SELECT order_status, COUNT(*) AS qtd
FROM mvp_olist.bronze.orders
GROUP BY order_status ORDER BY qtd DESC;

order_status,qtd
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
%sql
SELECT
  SUM(CASE WHEN try_cast(price AS DOUBLE) IS NULL THEN 1 ELSE 0 END)  AS preco_invalido,
  SUM(CASE WHEN try_cast(price AS DOUBLE) <= 0 THEN 1 ELSE 0 END)     AS preco_zero_ou_negativo,
  MIN(try_cast(price AS DOUBLE))                                      AS preco_min,
  percentile_approx(try_cast(price AS DOUBLE), 0.5)                   AS preco_mediana,
  percentile_approx(try_cast(price AS DOUBLE), 0.99)                  AS preco_p99,
  MAX(try_cast(price AS DOUBLE))                                      AS preco_max,
  MIN(try_cast(freight_value AS DOUBLE))                              AS frete_min,
  MAX(try_cast(freight_value AS DOUBLE))                              AS frete_max
FROM mvp_olist.bronze.order_items;

preco_invalido,preco_zero_ou_negativo,preco_min,preco_mediana,preco_p99,preco_max,frete_min,frete_max
0,0,0.85,74.99,889.99,6735.0,0.0,409.68


## Conclusões do diagnóstico
| Achado | Tratamento |
|---|---|
| 610 produtos sem categoria | preenchido com `sem_categoria` |
| 2 categorias sem tradução | fallback para o nome em português |
| Colunas de nome/descrição/fotos com nulos e erro de digitação | removidas (fora do escopo) |
| Pedidos sem data de entrega/aprovação | mantidos nulos (pedidos não entregues) |
| Duplicatas e órfãos | nenhum encontrado |
| Pedidos canceled/unavailable | flag `is_venda_valida = false` |
| Preços altos (até R$ 6.735) e frete R$ 0 | mantidos (valores reais) |
